# Pakistan Transport Fare Prediction

Generates synthetic Pakistan transport data, trains Linear Regression and Random Forest models, evaluates them, and saves the best model as `fare_model.pkl`.

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

np.random.seed(42)

## 1. Generate Pakistan transport dataset (1000 rows)

In [ ]:
N = 1000

time_of_day_options = ['morning', 'afternoon', 'evening', 'night']
route_type_options = ['city', 'intercity', 'highway']

time_of_day = np.random.choice(time_of_day_options, size=N, p=[0.35, 0.25, 0.30, 0.10])
route_type = np.random.choice(route_type_options, size=N, p=[0.5, 0.3, 0.2])

distance_km = np.where(
    route_type == 'city',
    np.random.uniform(1, 25, N),
    np.where(
        route_type == 'intercity',
        np.random.uniform(40, 350, N),
        np.random.uniform(100, 800, N),
    ),
)

passengers = np.random.randint(1, 6, size=N)

# Base fare (PKR) per route type
base_rate = np.where(route_type == 'city', 40,
             np.where(route_type == 'intercity', 25, 18))

# Time of day multiplier (rush hours cost more)
tod_mult = np.where(time_of_day == 'morning', 1.15,
            np.where(time_of_day == 'evening', 1.20,
             np.where(time_of_day == 'night', 1.30, 1.00)))

fare_pkr = (
    50
    + distance_km * base_rate * tod_mult
    + passengers * 15
    + np.random.normal(0, 30, N)
)
fare_pkr = np.round(np.clip(fare_pkr, 50, None), 2)

df = pd.DataFrame({
    'distance_km': np.round(distance_km, 2),
    'time_of_day': time_of_day,
    'route_type': route_type,
    'passengers': passengers,
    'fare_pkr': fare_pkr,
})

df.head()

## 2. Clean the data

In [ ]:
print('Shape:', df.shape)
print('Missing values:\n', df.isna().sum())
print('Duplicates:', df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

# Remove outliers via IQR on fare
q1, q3 = df['fare_pkr'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df = df[(df['fare_pkr'] >= low) & (df['fare_pkr'] <= high)].reset_index(drop=True)

print('Cleaned shape:', df.shape)
df.describe()

## 3. Visualisations

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='distance_km', y='fare_pkr', hue='route_type', alpha=0.6)
plt.title('Fare vs Distance by Route Type')
plt.xlabel('Distance (km)')
plt.ylabel('Fare (PKR)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='time_of_day', y='fare_pkr',
            order=['morning', 'afternoon', 'evening', 'night'])
plt.title('Fare Distribution by Time of Day')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
avg = df.groupby('route_type')['fare_pkr'].mean().sort_values()
sns.barplot(x=avg.index, y=avg.values)
plt.title('Average Fare by Route Type')
plt.ylabel('Average Fare (PKR)')
plt.tight_layout()
plt.show()

## 4. Encode features

In [ ]:
TIME_MAP = {'morning': 0, 'afternoon': 1, 'evening': 2, 'night': 3}
ROUTE_MAP = {'city': 0, 'intercity': 1, 'highway': 2}

data = df.copy()
data['time_of_day'] = data['time_of_day'].map(TIME_MAP)
data['route_type'] = data['route_type'].map(ROUTE_MAP)

X = data[['distance_km', 'time_of_day', 'route_type', 'passengers']].values
y = data['fare_pkr'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 5. Train and evaluate models

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_r2 = r2_score(y_test, lr_pred)
print(f'Linear Regression -> MAE: {lr_mae:.2f}  R2: {lr_r2:.4f}')

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)
print(f'Random Forest     -> MAE: {rf_mae:.2f}  R2: {rf_r2:.4f}')

## 6. Save the best model

In [ ]:
best_model, best_name = (rf, 'RandomForest') if rf_r2 >= lr_r2 else (lr, 'LinearRegression')
joblib.dump(best_model, 'fare_model.pkl')
print(f'Saved best model ({best_name}) to fare_model.pkl')

## 7. Sample prediction

In [ ]:
sample = np.array([[15.0, TIME_MAP['morning'], ROUTE_MAP['city'], 2]])
print('Predicted fare (PKR):', round(float(best_model.predict(sample)[0]), 2))